In [ ]:
import speech_recognition as sr
from pathlib import Path
import wave
import numpy as np
import sounddevice as sd
import soundfile as sf
import time
import sys
import os

class SpeechRecognitionSystem:
    def __init__(self):
        """Initialize the speech recognition system."""
        try:
            self.recognizer = sr.Recognizer()
            # Adjust recognition sensitivity
            self.recognizer.energy_threshold = 300
            self.recognizer.dynamic_energy_threshold = True
            self.recognizer.pause_threshold = 0.8
            print("Speech recognition system initialized successfully.")
        except Exception as e:
            raise Exception(f"Failed to initialize speech recognition: {str(e)}")
        
    def record_audio(self, duration=5, sample_rate=44100):
        """Record audio from microphone."""
        print(f"\nPreparing to record for {duration} seconds...")
        print("3...")
        time.sleep(1)
        print("2...")
        time.sleep(1)
        print("1...")
        time.sleep(1)
        print("Recording now! Please speak...")
        
        try:
            # Record audio
            recording = sd.rec(
                int(duration * sample_rate),
                samplerate=sample_rate,
                channels=1,
                dtype=np.int16
            )
            
            # Wait for recording to complete
            sd.wait()
            
            # Create output directory if it doesn't exist
            os.makedirs('recordings', exist_ok=True)
            
            # Save recording with timestamp
            timestamp = time.strftime("%Y%m%d-%H%M%S")
            output_path = f"recordings/recording_{timestamp}.wav"
            sf.write(output_path, recording, sample_rate)
            print(f"\nAudio saved to: {output_path}")
            
            return output_path
        except Exception as e:
            raise Exception(f"Recording failed: {str(e)}")

    def transcribe_file(self, audio_path):
        """Transcribe audio from a file."""
        try:
            # Check if file exists
            if not Path(audio_path).exists():
                raise FileNotFoundError(f"Audio file not found: {audio_path}")
            
            # Load audio file
            with sr.AudioFile(audio_path) as source:
                print("\nProcessing audio file...")
                audio = self.recognizer.record(source)
                
                print("Transcribing... This may take a moment.")
                
                try:
                    # Try using Google's speech recognition
                    text = self.recognizer.recognize_google(audio)
                    return text
                except sr.RequestError:
                    print("\nOnline recognition unavailable, falling back to offline mode...")
                    try:
                        text = self.recognizer.recognize_sphinx(audio)
                        return text
                    except Exception as e:
                        raise Exception("Offline recognition failed. Please check your internet connection or try again.")
                        
        except Exception as e:
            raise Exception(f"Transcription failed: {str(e)}")

    def transcribe_microphone(self, duration=5):
        """Record and transcribe audio from microphone."""
        try:
            # Record audio
            audio_path = self.record_audio(duration)
            
            # Transcribe recorded audio
            text = self.transcribe_file(audio_path)
            
            return text, audio_path
            
        except Exception as e:
            raise Exception(f"Microphone transcription failed: {str(e)}")

    def analyze_audio_file(self, audio_path):
        """Analyze audio file properties."""
        try:
            with wave.open(audio_path, 'rb') as wave_file:
                properties = {
                    'channels': wave_file.getnchannels(),
                    'sample_width': wave_file.getsampwidth(),
                    'frame_rate': wave_file.getframerate(),
                    'frames': wave_file.getnframes(),
                    'duration': round(wave_file.getnframes() / wave_file.getframerate(), 2)
                }
                return properties
        except Exception as e:
            raise Exception(f"Audio analysis failed: {str(e)}")

def clear_screen():
    """Clear the console screen."""
    os.system('cls' if os.name == 'nt' else 'clear')

def get_menu_choice():
    """Get user menu choice with input validation."""
    while True:
        try:
            print("\nSpeech Recognition Menu:")
            print("1. Record and transcribe from microphone")
            print("2. Transcribe existing audio file")
            print("3. Analyze audio file properties")
            print("4. Exit")
            
            choice = input("\nEnter your choice (1-4): ").strip()
            
            if choice in ['1', '2', '3', '4']:
                return choice
            else:
                print("\nInvalid choice. Please enter a number between 1 and 4.")
        except Exception:
            print("\nInvalid input. Please try again.")

def get_duration():
    """Get recording duration with input validation."""
    while True:
        try:
            duration = int(input("\nEnter recording duration in seconds (1-60): ").strip())
            if 1 <= duration <= 60:
                return duration
            else:
                print("Please enter a duration between 1 and 60 seconds.")
        except ValueError:
            print("Please enter a valid number.")

def get_file_path():
    """Get and validate file path."""
    while True:
        path = input("\nEnter path to audio file: ").strip()
        if path.lower() == 'cancel':
            return None
        if os.path.exists(path):
            return path
        print(f"File not found: {path}")
        print("Please enter a valid path or 'cancel' to go back.")

def main():
    """Main function demonstrating the speech recognition system."""
    
    try:
        print("Initializing speech recognition system...")
        system = SpeechRecognitionSystem()
        
        while True:
            choice = get_menu_choice()
            
            if choice == '1':
                try:
                    duration = get_duration()
                    text, audio_path = system.transcribe_microphone(duration)
                    print("\nTranscription Results:")
                    print("-" * 50)
                    print(f"Audio file: {audio_path}")
                    print(f"Transcribed text: {text}")
                    print("-" * 50)
                    input("\nPress Enter to continue...")
                except Exception as e:
                    print(f"\nError: {str(e)}")
                    input("\nPress Enter to continue...")
                    
            elif choice == '2':
                audio_path = get_file_path()
                if audio_path:
                    try:
                        text = system.transcribe_file(audio_path)
                        print("\nTranscription Results:")
                        print("-" * 50)
                        print(f"Transcribed text: {text}")
                        print("-" * 50)
                        input("\nPress Enter to continue...")
                    except Exception as e:
                        print(f"\nError: {str(e)}")
                        input("\nPress Enter to continue...")
                    
            elif choice == '3':
                audio_path = get_file_path()
                if audio_path:
                    try:
                        properties = system.analyze_audio_file(audio_path)
                        print("\nAudio Properties:")
                        print("-" * 50)
                        for key, value in properties.items():
                            print(f"{key.replace('_', ' ').title()}: {value}")
                        print("-" * 50)
                        input("\nPress Enter to continue...")
                    except Exception as e:
                        print(f"\nError: {str(e)}")
                        input("\nPress Enter to continue...")
                    
            elif choice == '4':
                print("\nThank you for using the Speech Recognition System!")
                break
            
            clear_screen()
                
    except KeyboardInterrupt:
        print("\nProgram terminated by user.")
        sys.exit(0)
        
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        sys.exit(1)

if __name__ == "__main__":
    main()

Initializing speech recognition system...
Speech recognition system initialized successfully.

Speech Recognition Menu:
1. Record and transcribe from microphone
2. Transcribe existing audio file
3. Analyze audio file properties
4. Exit
